# TimeGAN Downstream Generator (Artifact-based)

This notebook is for downstream data generation only.

It loads a trained TimeGAN artifact and exports:
- Independent synthetic windows
- A stitched synthetic block (concatenated windows)
- Optional stitched + real combined dataset

## What can be  change
1. In **Cell 3** (`CFG`):
   - artifact paths
   - `n_independent_windows`
   - stitched length (`stitch_total_months` or `stitch_num_windows`)
   - whether to prepend synthetic before real
2. Run cells top-to-bottom.

##  note
- `sequence_length` is fixed by the trained model artifact.
- To change per-window length (e.g., from 24 to 60), retrain in the research notebook first.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer

In [ ]:
CFG = {
    # --------- artifact paths ---------
    'model_artifact': '../data/timegan_outputs/timegan_model.pkl',
    'scaler_artifact': '../data/timegan_outputs/timegan_scaler.pkl',
    'meta_artifact': '../data/timegan_outputs/timegan_artifact_metadata.json',

    # --------- generation controls ---------
    # number of independent windows (each window has model sequence_length months)
    'n_independent_windows': 50,

    # stitched length control (set one of these)
    # Option A: desired stitched total months (recommended)
    'stitch_total_months': 312 * 24,

    # Option B: explicit number of windows (used if stitch_total_months is None)
    'stitch_num_windows': 312,

    # if True, synthetic block is dated to end one month before real data starts
    'stitch_prepend_before_real': True,

    # export synthetic+real combined table
    'export_stitched_with_real': True,

    # --------- output ---------
    'output_dir': '../data/timegan_outputs'
}

Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
CFG

### Config guide
- `n_independent_windows`: number of standalone windows to generate.
- `stitch_total_months`: target total stitched length in months.
- `stitch_num_windows`: fallback if `stitch_total_months=None`.
- `stitch_prepend_before_real=True`: synthetic dates come before real history.

In [ ]:
def sample_synthetic(synth, scaler, n_samples, seq_len, n_features):
    s = np.asarray(synth.sample(n_samples))
    if s.ndim != 3:
        raise ValueError(f'Unexpected synthetic shape: {s.shape}')
    return scaler.inverse_transform(s.reshape(-1, n_features)).reshape(-1, seq_len, n_features)

In [ ]:
with open(CFG['meta_artifact'], 'r') as f:
    meta = json.load(f)

with open(CFG['scaler_artifact'], 'rb') as f:
    scaler = pickle.load(f)

synth_model = TimeSeriesSynthesizer.load(CFG['model_artifact'])

feature_cols = meta['feature_cols']
params = meta['params']
data_path = meta['data_path']
date_col = meta['date_col']
seq_len = int(params['sequence_length'])

real_df = pd.read_csv(data_path)
real_df[date_col] = pd.to_datetime(real_df[date_col])
real_df = real_df.sort_values(date_col).reset_index(drop=True)

# Resolve stitched window count from user-friendly controls
if CFG.get('stitch_total_months') is not None:
    n_stitch = int(np.ceil(CFG['stitch_total_months'] / seq_len))
else:
    n_stitch = int(CFG['stitch_num_windows'])

print('Loaded artifact params:', params)
print('Features:', feature_cols)
print(f'Window length from model: {seq_len} months')
print(f'Independent windows to generate: {CFG["n_independent_windows"]}')
print(f'Stitched windows to generate: {n_stitch}')
print(f'Approx stitched months: {n_stitch * seq_len}')

In [ ]:
# 1) Independent synthetic windows (long format with sample_id/step)
final_synth_w = sample_synthetic(
    synth_model, scaler,
    n_samples=int(CFG['n_independent_windows']),
    seq_len=seq_len,
    n_features=len(feature_cols)
)

records = []
for i in range(final_synth_w.shape[0]):
    for t in range(final_synth_w.shape[1]):
        rec = {'sample_id': i + 1, 'step': t + 1}
        for j, c in enumerate(feature_cols):
            rec[c] = float(final_synth_w[i, t, j])
        records.append(rec)

synth_long_df = pd.DataFrame(records)
out_synth = Path(CFG['output_dir']) / 'timegan_synthetic_sequences_long.csv'
synth_long_df.to_csv(out_synth, index=False)
print('Saved:', out_synth)
print(f'Independent output rows: {len(synth_long_df)} (= {CFG["n_independent_windows"]} x {seq_len})')
display(synth_long_df.head())

In [ ]:
# 2) Stitched synthetic block 
stitched_w = sample_synthetic(
    synth_model, scaler,
    n_samples=n_stitch,
    seq_len=seq_len,
    n_features=len(feature_cols)
)
stitched_flat = stitched_w.reshape(-1, len(feature_cols))
stitched_df = pd.DataFrame(stitched_flat, columns=feature_cols)

if CFG.get('stitch_prepend_before_real', True):
    real_start = pd.to_datetime(real_df[date_col].min())
    stitched_dates = pd.date_range(
        end=real_start - pd.DateOffset(months=1),
        periods=len(stitched_df),
        freq='MS'
    )
    stitched_df.insert(0, date_col, stitched_dates)

out_stitched_only = Path(CFG['output_dir']) / 'timegan_stitched_synthetic_only.csv'
stitched_df.to_csv(out_stitched_only, index=False)
print('Saved:', out_stitched_only)
print(f'Stitched months: {len(stitched_df)} (= {n_stitch} x {seq_len})')
display(stitched_df.head())

In [ ]:
# 3) Optional stitched + real combined dataset
if CFG.get('export_stitched_with_real', True):
    real_df_for_concat = real_df[[date_col] + feature_cols].copy()
    if CFG.get('stitch_prepend_before_real', True):
        stitched_with_real = pd.concat([stitched_df, real_df_for_concat], axis=0, ignore_index=True)
    else:
        stitched_with_real = pd.concat([real_df_for_concat, stitched_df], axis=0, ignore_index=True)

    stitched_with_real = stitched_with_real.sort_values(date_col).reset_index(drop=True)
    out_stitched_with_real = Path(CFG['output_dir']) / 'timegan_stitched_with_real.csv'
    stitched_with_real.to_csv(out_stitched_with_real, index=False)
    print('Saved:', out_stitched_with_real)
    display(stitched_with_real.head())

### Output files
- `timegan_synthetic_sequences_long.csv`: independent windows in long format (`sample_id`, `step`).
- `timegan_stitched_synthetic_only.csv`: concatenated synthetic block.
- `timegan_stitched_with_real.csv`: stitched synthetic + real history (if enabled).